# 🎯 Bowling Analysis - IPL (2008-2025)

Comprehensive analysis of bowling statistics including wickets, economy rates, and bowler performance trends.

In [1]:
import pathlib
import sys


def find_project_root(start: pathlib.Path) -> pathlib.Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    return start


ROOT_DIR = find_project_root(pathlib.Path.cwd().resolve())
sys.path.append(str(ROOT_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from src.data_loader import load_data
from src.analytics.bowling import (
    bowler_summary, wiickets_by_season, wicket_againt_teams,
    wickets_by_venue, best_bowling_figures, bowling_milestones
)

# Set styling
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

# Load data
matches, deliveries = load_data()
print("✅ Bowling Data loaded successfully!")
print(f"Matches: {len(matches)}, Deliveries: {len(deliveries)}")


✅ Bowling Data loaded successfully!
Matches: 1169, Deliveries: 278205


## 1. Top Wicket Takers - All Time

In [2]:
# Top 15 wicket takers
wickets_by_bowler = deliveries[deliveries['player_dismissed'].notna()].groupby('bowler').size().sort_values(ascending=False).head(15)

fig = px.bar(
    x=wickets_by_bowler.values,
    y=wickets_by_bowler.index,
    orientation='h',
    title='Top 15 Wicket Takers in IPL (2008-2025)',
    labels={'x': 'Wickets', 'y': 'Bowler'},
    color=wickets_by_bowler.values,
    color_continuous_scale='Plasma'
)
fig.update_layout(height=500, showlegend=False)
fig.show()

print("🎳 Top 15 Wicket Takers:")
print(wickets_by_bowler)

🎳 Top 15 Wicket Takers:
bowler
YS Chahal          229
B Kumar            213
SP Narine          212
DJ Bravo           207
R Ashwin           205
JJ Bumrah          203
PP Chawla          201
SL Malinga         188
A Mishra           183
RA Jadeja          179
HV Patel           168
Rashid Khan        166
UT Yadav           163
Sandeep Sharma     163
Harbhajan Singh    161
dtype: int64


## 2. Economy Rate Analysis

In [3]:
# Calculate economy rate (min 200 balls bowled)
bowler_balls = deliveries.groupby('bowler').size()
bowler_runs = deliveries.groupby('bowler')[['batsman_runs', 'extras']].sum().sum(axis=1)
economy = (bowler_runs / (bowler_balls / 6)).sort_values()

# Filter bowlers with at least 200 balls (≈33 overs)
economy_filtered = economy[bowler_balls >= 200].head(15)

fig = px.bar(
    x=economy_filtered.values,
    y=economy_filtered.index,
    orientation='h',
    title='Top 15 Most Economical Bowlers (Min 200 balls)',
    labels={'x': 'Economy Rate', 'y': 'Bowler'},
    color=economy_filtered.values,
    color_continuous_scale='RdYlGn_r'
)
fig.update_layout(height=500, showlegend=False)
fig.show()

print("💨 Best Economy Rates (Min 200 balls):")
print(economy_filtered)


💨 Best Economy Rates (Min 200 balls):
bowler
Sohail Tanvir       6.226415
A Chandila          6.282051
SM Pollock          6.578571
A Kumble            6.646999
GD McGrath          6.674772
M Muralitharan      6.698292
J Yadav             6.738693
RE van der Merwe    6.791209
DW Steyn            6.791411
SP Narine           6.825153
DL Vettori          6.833121
R Rampaul           6.884892
J Botha             6.922426
SL Malinga          7.032952
Harbhajan Singh     7.038330
dtype: float64


## 3. Wickets Progression by Season

In [4]:
# Total wickets by season
season_lookup = matches[["matchId", "season"]].drop_duplicates()
deliveries_with_season = deliveries.merge(season_lookup, on="matchId", how="left")
wickets_by_season = (
    deliveries_with_season.dropna(subset=["season"])
    .loc[deliveries_with_season["player_dismissed"].notna()]
    .groupby("season")
    .size()
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=wickets_by_season.index,
    y=wickets_by_season.values,
    mode="lines+markers",
    name="Total Wickets",
    line=dict(color="#ff7f0e", width=3),
    marker=dict(size=8)
))

fig.update_layout(
    title="Total Wickets by Season",
    xaxis_title="Season",
    yaxis_title="Wickets",
    height=500,
    hovermode="x unified"
)
fig.show()

print("📊 Wickets by Season:")
print(wickets_by_season)


📊 Wickets by Season:
season
2007/08    690
2009       698
2009/10    725
2011       813
2012       858
2013       912
2014       674
2015       691
2016       666
2017       711
2018       722
2019       685
2020/21    677
2021       717
2022       912
2023       916
2024       883
2025       873
dtype: int64


## 4. Dismissal Type Analysis

In [5]:
# Dismissal types
dismissal_types = deliveries[deliveries['player_dismissed'].notna()]['dismissal_kind'].value_counts()

fig = px.pie(
    values=dismissal_types.values,
    names=dismissal_types.index,
    title='Distribution of Dismissal Types',
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig.show()

print("🎯 Dismissal Type Distribution:")
print(dismissal_types)
print(f"\nTotal Dismissals: {dismissal_types.sum()}")

🎯 Dismissal Type Distribution:
dismissal_kind
caught                   8665
bowled                   2345
run out                  1153
lbw                       853
caught and bowled         388
stumped                   376
hit wicket                 18
retired hurt               17
retired out                 5
obstructing the field       3
Name: count, dtype: int64

Total Dismissals: 13823


## 5. Best Bowling Figures

In [6]:
# Best bowling figures (runs conceded and wickets in an innings)
deliveries_with_runs = deliveries.assign(runs=deliveries['batsman_runs'] + deliveries['extras'])
bowling_by_match = (
    deliveries_with_runs.groupby(['matchId', 'bowler'])
    .agg(wickets=('player_dismissed', lambda x: x.notna().sum()), runs=('runs', 'sum'))
    .reset_index()
)

bowling_by_match['figure'] = bowling_by_match['wickets'].astype(str) + '/' + bowling_by_match['runs'].astype(str)

best_figures = bowling_by_match.nlargest(10, 'wickets')[['bowler', 'wickets', 'runs', 'figure']]

fig = px.bar(
    best_figures,
    x='bowler',
    y='wickets',
    title='Top 10 Best Bowling Performances (by Wickets)',
    labels={'bowler': 'Bowler', 'wickets': 'Wickets', 'runs': 'Runs'},
    hover_data=['figure', 'runs'],
    color='wickets',
    color_continuous_scale='Viridis'
)
fig.update_layout(height=500, showlegend=False)
fig.show()

print("🏆 Best Bowling Performances:")
print(best_figures)


🏆 Best Bowling Performances:
              bowler  wickets  runs figure
290    Sohail Tanvir        6    15   6/15
4546       DJG Sammy        6    23   6/23
6510      AD Russell        6    25   6/25
6609         A Zampa        6    19   6/19
8499       AS Joseph        6    14   6/14
9668        HV Patel        6    27   6/27
11969        B Kumar        6    31   6/31
12086  Akash Madhwal        6     6    6/6
13092       MA Starc        6    35   6/35
364     CRD Fernando        5    18   5/18


## 6. Multi-Wicket Hauls

In [7]:
# Multi-wicket hauls
three_w = bowling_by_match[bowling_by_match['wickets'] >= 3].groupby('bowler').size().sort_values(ascending=False).head(10)
five_w = bowling_by_match[bowling_by_match['wickets'] >= 5].groupby('bowler').size()

hauls_data = pd.DataFrame({
    '3-Wicket Hauls': three_w,
    '5-Wicket Hauls': five_w
}).fillna(0).astype(int)

fig = px.bar(
    hauls_data,
    title='Bowlers with 3-Wicket & 5-Wicket Hauls',
    barmode='group',
    color_discrete_map={'3-Wicket Hauls': '#1f77b4', '5-Wicket Hauls': '#d62728'}
)
fig.update_layout(height=500, showlegend=True)
fig.show()

print("🎳 Multi-Wicket Hauls:")
print(hauls_data)

🎳 Multi-Wicket Hauls:
                   3-Wicket Hauls  5-Wicket Hauls
bowler                                           
A Kumble                        0               2
A Mishra                       20               1
A Nehra                        18               1
A Zampa                         0               1
AD Mascarenhas                  0               1
AD Russell                      0               2
AJ Tye                          0               2
AS Joseph                       0               1
AS Rajpoot                      0               1
Akash Madhwal                   0               1
Arshdeep Singh                  0               1
B Kumar                         0               2
BJ Hodge                        0               1
CH Morris                       0               1
CRD Fernando                    0               1
CV Varun                        0               1
DJ Bravo                       21               0
DJG Sammy                   